# Sequential minimum sensor budgets

Budget trends for paper §3.4.


## Saved-result reuse

`REUSE_SAVED_RESULTS=True` reuses compatible DP results; missing cases are solved. Notebook 01 is optional.


In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

working_directory = Path.cwd().resolve()
PROJECT_DIRECTORY = (
    working_directory / "problem-1-zero-error"
    if (working_directory / "problem-1-zero-error").is_dir()
    else working_directory
)
SOURCE_DIRECTORY = PROJECT_DIRECTORY / "src"
if str(SOURCE_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIRECTORY))
for module in ("run_config", "sensor_budget_analysis", "global_minimum", "phase_data", "optimal", "eval"):
    sys.modules.pop(module, None)
from sensor_budget_analysis import (
    run_budget_analysis, plot_budget_comparison, configure_style, save_figure,
)
print("Project directory:", PROJECT_DIRECTORY)


## Configuration

Choose `RUN_MODE="test"` or `"full"`; final figures use full results.


In [ ]:
RUN_MODE = "full"  # "test" or "full"
REUSE_SAVED_RESULTS = True  # False forces a fresh DP solve for every selected case.

if RUN_MODE not in ("test", "full"):
    raise ValueError('RUN_MODE must be "test" or "full"')
print("Run mode:", RUN_MODE)
print("Expected standard benchmark cases:", 132 if RUN_MODE == "test" else 39_600)


## Obtain sequential optima


In [ ]:
analysis = run_budget_analysis(PROJECT_DIRECTORY, run_mode=RUN_MODE, reuse_saved=REUSE_SAVED_RESULTS)
nmin_rows = analysis["summary"]
OUTPUT_DIRECTORY = analysis["output_directory"]
print("Cases:", len(analysis["rows"]))
print("Output:", OUTPUT_DIRECTORY)

## Budget summary


In [ ]:
def show_summary(rows, include_global=False):
    header = r"| Dimension | K | Cases | Mean $N_{\min}^{\mathrm{seq}}$ |"
    separator = "|---|---:|---:|---:|"
    if include_global:
        header += r" Mean $N_{\min}^{\mathrm{global}}$ | Mean gap | Equal budgets (%) |"
        separator += "---:|---:|---:|"
    lines = [header, separator]
    for row in rows:
        line = (f"| {row['dimension']} | {row['K']} | {row['case_count']} | "
                f"{row['mean_N_min_opt']:.3f} |")
        if include_global:
            line += (f" {row['mean_N_min_global']:.3f} | {row['mean_sensor_gap']:.3f} | "
                     f"{100 * row['equal_budget_fraction']:.1f} |")
        lines.append(line)
    display(Markdown("\n".join(lines)))

show_summary(nmin_rows)

## Budget trends


In [ ]:
configure_style()
figure, axis = plt.subplots(figsize=(6.4, 4.0))
for dimension, color, marker, line in ((1, "#0072B2", "o", "-"), (2, "#009E73", "s", ":")):
    selected = [row for row in nmin_rows if row["spatial_dimension"] == dimension]
    axis.plot([row["K"] for row in selected], [row["mean_N_min_opt"] for row in selected],
              label=f"{dimension}D", color=color, marker=marker, linestyle=line, linewidth=1.8)
axis.set_xlabel(r"Number of candidate fields, $K$")
axis.set_ylabel(r"Mean exact sensor budget, $N_{\min}^{\mathrm{seq}}$")
axis.set_xticks(sorted({row["K"] for row in nmin_rows}))
axis.grid(alpha=0.22)
axis.legend(frameon=False)
if RUN_MODE == "test":
    axis.set_title(f"Test run: {len(analysis['rows']):,} cases")
figure.tight_layout()
save_figure(figure, OUTPUT_DIRECTORY / "figures", "benchmark_nmin_by_k")
display(figure)
plt.close(figure)

## Outputs

Per-case budgets and summaries: `results/nmin/{test,full}/tables/`.
